# Structure of the legacy securities lending data

Source table `crp_sftds_ecb_legacy.state_sl_079`, the trade states before June 2026, to be used from 2021-01-01 to 2026-05-31. Same ESMA report as the new table, stored the old way, with the collateral as extra rows per piece via `local_index` instead of arrays. Five checks before the cleaning query is mapped onto it. Single day checks use 2026-05-29, the last business day of May 2026, and rely on the table being partitioned by `business_date`.

In [1]:
import pyodbc
import pandas as pd
import numpy as np

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 250)

Which column is the table partitioned by? Single day filters are only cheap if it is `business_date`.

In [3]:
query = f"""

SHOW PARTITIONS crp_sftds_ecb_legacy.state_sl_079

"""
df = pd.read_sql_query(query, cnxn)
df.tail(3)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_29564\591883660.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,db_type,business_date,tec_execution_date,#Rows,#Files,Size,Bytes Cached,Cache Replication,Format,Incremental stats,Location,EC Policy
1562,ECB,2026-07-22,20260728181837,5862581,20,2.44GB,NOT CACHED,NOT CACHED,PARQUET,true,s3a://devo-crp-2gn5maj9/sftds_ecb_legacy/db/st...,NONE
1563,ECB,2026-07-23,20260728211700,6053480,20,2.50GB,NOT CACHED,NOT CACHED,PARQUET,true,s3a://devo-crp-2gn5maj9/sftds_ecb_legacy/db/st...,NONE
1564,Total,,,40817302,25822,2.70TB,0B,,,,,


## 1. What is a row, and what identifies a report?

Every report should have a row with `local_index = 0`, exactly one, and `techrcrdid` should be the same on all rows of a report.

In [2]:
query = f"""

SELECT local_index, COUNT(*) AS n
FROM crp_sftds_ecb_legacy.state_sl_079
WHERE business_date = '2026-05-29'
GROUP BY 1
ORDER BY 1
LIMIT 25

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_29564\4001327716.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,local_index,n
0,0,2159306
1,1,324573
2,2,90592
3,3,85450
4,4,68598
5,5,63294
6,6,60923
7,7,59094
8,8,57153
9,9,56395


In [4]:
query = f"""

SELECT COUNT(*) AS n_rows,
       COUNT(DISTINCT techrcrdid) AS n_reports,
       SUM(CASE WHEN local_index = 0 THEN 1 ELSE 0 END) AS n_row0,
       COUNT(DISTINCT CASE WHEN local_index = 0 THEN techrcrdid END) AS n_reports_with_row0,
       COUNT(DISTINCT tec_surrogate_key) AS n_surrogate_keys
FROM crp_sftds_ecb_legacy.state_sl_079
WHERE business_date = '2026-05-29'

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_29564\1334257864.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,n_rows,n_reports,n_row0,n_reports_with_row0,n_surrogate_keys
0,5631954,2159203,2159306,2159203,5631954


## 2. Which codes do the categorical fields use?

`direction` replaces `counterparty_side`, `is_opn_term` replaces `term_type`, `uncollsd` is a string where the new table has a boolean.

In [5]:
query = f"""

SELECT 'direction' AS col, CAST(direction AS STRING) AS value, COUNT(*) AS n
FROM crp_sftds_ecb_legacy.state_sl_079 WHERE business_date = '2026-05-29' AND local_index = 0 GROUP BY 1, 2
UNION ALL
SELECT 'is_opn_term', CAST(is_opn_term AS STRING), COUNT(*)
FROM crp_sftds_ecb_legacy.state_sl_079 WHERE business_date = '2026-05-29' AND local_index = 0 GROUP BY 1, 2
UNION ALL
SELECT 'uncollsd', CAST(uncollsd AS STRING), COUNT(*)
FROM crp_sftds_ecb_legacy.state_sl_079 WHERE business_date = '2026-05-29' AND local_index = 0 GROUP BY 1, 2
UNION ALL
SELECT 'rbtrate_type', CAST(rbtrate_type AS STRING), COUNT(*)
FROM crp_sftds_ecb_legacy.state_sl_079 WHERE business_date = '2026-05-29' AND local_index = 0 GROUP BY 1, 2
UNION ALL
SELECT 'ctrctmod_lvl', CAST(ctrctmod_lvl AS STRING), COUNT(*)
FROM crp_sftds_ecb_legacy.state_sl_079 WHERE business_date = '2026-05-29' AND local_index = 0 GROUP BY 1, 2
UNION ALL
SELECT 'ctrctmod_actntp', CAST(ctrctmod_actntp AS STRING), COUNT(*)
FROM crp_sftds_ecb_legacy.state_sl_079 WHERE business_date = '2026-05-29' AND local_index = 0 GROUP BY 1, 2
ORDER BY col, n DESC

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_29564\3348371305.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


DatabaseError: Execution failed on sql '

SELECT 'direction' AS col, CAST(direction AS STRING) AS value, COUNT(*) AS n
FROM crp_sftds_ecb_legacy.state_sl_079 WHERE business_date = '2026-05-29' AND local_index = 0 GROUP BY 1, 2
UNION ALL
SELECT 'is_opn_term', CAST(is_opn_term AS STRING), COUNT(*)
FROM crp_sftds_ecb_legacy.state_sl_079 WHERE business_date = '2026-05-29' AND local_index = 0 GROUP BY 1, 2
UNION ALL
SELECT 'uncollsd', CAST(uncollsd AS STRING), COUNT(*)
FROM crp_sftds_ecb_legacy.state_sl_079 WHERE business_date = '2026-05-29' AND local_index = 0 GROUP BY 1, 2
UNION ALL
SELECT 'rbtrate_type', CAST(rbtrate_type AS STRING), COUNT(*)
FROM crp_sftds_ecb_legacy.state_sl_079 WHERE business_date = '2026-05-29' AND local_index = 0 GROUP BY 1, 2
UNION ALL
SELECT 'ctrctmod_lvl', CAST(ctrctmod_lvl AS STRING), COUNT(*)
FROM crp_sftds_ecb_legacy.state_sl_079 WHERE business_date = '2026-05-29' AND local_index = 0 GROUP BY 1, 2
UNION ALL
SELECT 'ctrctmod_actntp', CAST(ctrctmod_actntp AS STR

## 3. How are quantity and price stored?

The legacy table splits units and nominal, and monetary, percentage and yield prices, into separate columns. The notation type of the new table has to be derived from which column is filled, using the new table's codes.

In [6]:
query = f"""

SELECT CASE WHEN lndata_assttp_scty_qty IS NOT NULL THEN 1 ELSE 0 END AS has_units,
       CASE WHEN lndata_assttp_scty_nmnl_amt IS NOT NULL THEN 1 ELSE 0 END AS has_nominal,
       CASE WHEN lndata_assttp_scty_unitpric_amt IS NOT NULL THEN 1 ELSE 0 END AS has_price_amount,
       CASE WHEN lndata_assttp_scty_unitpric_pctg IS NOT NULL THEN 1 ELSE 0 END AS has_price_pct,
       CASE WHEN lndata_assttp_scty_unitpric_yld IS NOT NULL THEN 1 ELSE 0 END AS has_price_yield,
       COUNT(*) AS n
FROM crp_sftds_ecb_legacy.state_sl_079
WHERE business_date = '2026-05-29' AND local_index = 0
GROUP BY 1, 2, 3, 4, 5
ORDER BY n DESC

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_29564\2545733726.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,has_units,has_nominal,has_price_amount,has_price_pct,has_price_yield,n
0,1,0,1,0,0,1429108
1,0,1,0,1,0,507417
2,0,0,0,0,0,131995
3,0,1,1,0,0,85005
4,1,0,0,1,0,4150
5,0,0,1,0,0,1596
6,0,0,0,1,0,35


The codes the new table uses for the same distinction.

In [7]:
query = f"""

SELECT loan_security_quantity_or_nominal_amount_notation_type AS quantity_notation,
       loan_security_price_notation_type AS price_notation,
       COUNT(*) AS n
FROM crp_sftds_ecb.trade_states_securitieslending
WHERE reference_period = (SELECT MAX(reference_period) FROM crp_sftds_ecb.trade_states_securitieslending)
GROUP BY 1, 2
ORDER BY n DESC

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_29564\57967840.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,quantity_notation,price_notation,n
0,Quantity,Monetary value,1515437
1,Monetary value,Percentage,565627
2,None,None,133420
3,Monetary value,Monetary value,93643
4,Quantity,Percentage,6537
5,None,Monetary value,1593
6,None,Percentage,35


## 4. Are the collateral component counts per report?

`collcmpnttp_scty` and `collcmpnttp_csh` should equal the number of rows of a report that carry a security or a cash piece.

In [8]:
query = f"""

SELECT r.collcmpnttp_scty, r.n_sec_rows, r.collcmpnttp_csh, r.n_cash_rows, COUNT(*) AS n_reports
FROM (
  SELECT techrcrdid,
         MAX(collcmpnttp_scty) AS collcmpnttp_scty,
         MAX(collcmpnttp_csh) AS collcmpnttp_csh,
         SUM(CASE WHEN assttp_scty_id IS NOT NULL THEN 1 ELSE 0 END) AS n_sec_rows,
         SUM(CASE WHEN assttp_csh_amt IS NOT NULL THEN 1 ELSE 0 END) AS n_cash_rows
  FROM crp_sftds_ecb_legacy.state_sl_079
  WHERE business_date = '2026-05-29'
  GROUP BY techrcrdid
) r
GROUP BY 1, 2, 3, 4
ORDER BY n_reports DESC
LIMIT 30

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_29564\1641786653.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,collcmpnttp_scty,n_sec_rows,collcmpnttp_csh,n_cash_rows,n_reports
0,0,0,0,0,1606036
1,0,0,1,1,325142
2,0,0,1,2,65847
3,1,1,0,0,48574
4,1,2,0,0,22778
5,1,4,0,0,16483
6,1,16,0,0,15923
7,1,5,0,0,5211
8,1,3,0,0,4091
9,1,55,0,0,2488


## 5. Does the deduplication key behave as in the new table?

Same group check as in the data structure notebook, on the row 0 rows only so that collateral rows do not count as legs, and on one day per year, the last business day of May from 2021 to 2026, instead of the full five years.

In [9]:
query = f"""

SELECT n_rows, n_best, n_dates, COUNT(*) AS n_groups
FROM (
  SELECT ruti, business_date,
         COUNT(*) AS n_rows,
         SUM(CASE WHEN best_value_leg = 1 THEN 1 ELSE 0 END) AS n_best,
         COUNT(DISTINCT evtdt) AS n_dates
  FROM crp_sftds_ecb_legacy.state_sl_079
  WHERE business_date IN ('2021-05-31', '2022-05-31', '2023-05-31', '2024-05-31', '2025-05-30', '2026-05-29')
    AND local_index = 0
  GROUP BY ruti, business_date
) g
GROUP BY n_rows, n_best, n_dates
ORDER BY n_groups DESC

"""
df = pd.read_sql_query(query, cnxn)
df.head(30)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_29564\2601730562.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,n_rows,n_best,n_dates,n_groups
0,1,0,1,11081913
1,2,1,1,672065
2,2,1,2,152828
3,2,0,2,112836
4,2,0,1,77425
5,1,1,1,6
6,3,0,2,4
7,3,1,2,2
8,30624,0,483,1
9,131845,0,943,1


The rows without `ruti` should again be the net exposure collateral updates without UTI.

In [10]:
query = f"""

SELECT ctrctmod_actntp, CASE WHEN uti IS NULL THEN 1 ELSE 0 END AS uti_missing, COUNT(*) AS n
FROM crp_sftds_ecb_legacy.state_sl_079
WHERE business_date = '2026-05-29' AND local_index = 0
  AND (ruti IS NULL OR ruti = '')
GROUP BY 1, 2
ORDER BY n DESC

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_29564\3488041391.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,ctrctmod_actntp,uti_missing,n
0,COLU,1,131845
